# Private API HTTP readback probe v2
CPU only. GetDatasetStatus returned403 in v1 and is not retried. Authenticated file download uses the documented HTTP resolver and a project-local cache; all403 responses stop that capability.

In [ ]:
import hashlib
import json
import os
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

base = Path("/kaggle/working/hung-api-storage-probe-v2")
base.mkdir(exist_ok=False)
os.environ["KAGGLEHUB_CACHE"] = str(base / "readback-cache")
os.environ["DISABLE_KAGGLE_CACHE"] = "true"

In [ ]:
import kagglehub
from kagglehub.clients import BackendError, build_kaggle_client
from kagglesdk.kernels.types.kernels_api_service import (
    ApiGetAcceleratorQuotaStatisticsRequest,
    ApiGetKernelSessionStatusRequest,
    ApiSaveKernelRequest,
)
from requests.exceptions import HTTPError

stage = base / "upload"
stage.mkdir()
handle = "hungluongduc/hung-e2e-api-storage-probe-20261027-v2"
capabilities = {}
with build_kaggle_client() as client:
    response = client.kernels.kernels_api_client.get_accelerator_quota_statistics(
        ApiGetAcceleratorQuotaStatisticsRequest()
    )
    gpu = response.gpu_quota
    assert gpu is not None
    assert all(v is not None for v in (gpu.time_used, gpu.time_reserved, gpu.total_time_allowed))
    quota = {
        "observed_at_utc": datetime.now(timezone.utc).isoformat(),
        "used_seconds": gpu.time_used.total_seconds(),
        "reserved_seconds": gpu.time_reserved.total_seconds(),
        "total_seconds": gpu.total_time_allowed.total_seconds(),
        "pay_to_scale_enabled": gpu.is_pay_to_scale_enabled,
    }
    assert not quota["pay_to_scale_enabled"]
    request = ApiGetKernelSessionStatusRequest()
    request.user_name = "hungluongduc"
    request.kernel_slug = "hung-matched-shard0001-44b8532-20261027"
    request.version_label = "1"
    try:
        prior = client.kernels.kernels_api_client.get_kernel_session_status(request)
        capabilities["read_kernel_status"] = prior.status.name
    except HTTPError as error:
        if error.response.status_code != 403:
            raise
        capabilities["read_kernel_status"] = "FORBIDDEN; no retry or alternate status endpoint"
    print(json.dumps({"quota": quota, "capabilities": capabilities}), flush=True)
probe = {
    "schema": "hung-private-durable-storage-probe-v2",
    "code_sha": "44b8532959d76fb20691881539f991ca44983727",
    "scope": "Tiny control receipt only; no model execution or credentials",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
archive = stage / "probe.zip.bin"
with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_DEFLATED) as stream:
    stream.writestr("probe.json", json.dumps(probe, sort_keys=True))
expected = hashlib.sha256(archive.read_bytes()).hexdigest()
print(
    json.dumps(
        {"before_upload_sha256": expected, "bytes": archive.stat().st_size, "handle": handle}
    ),
    flush=True,
)
started = time.perf_counter()
kagglehub.dataset_upload(
    handle, str(stage), version_notes="Private create/readback probe v2; no GetDatasetStatus call"
)
upload_seconds = time.perf_counter() - started
readback_started = time.perf_counter()
for attempt in range(60):
    try:
        remote = Path(
            kagglehub.dataset_download(
                handle + "/versions/1", path="probe.zip.bin", force_download=True
            )
        )
        break
    except (HTTPError, BackendError) as error:
        response = getattr(error, "response", None)
        status = (
            response.status_code if response is not None else getattr(error, "error_code", None)
        )
        if status not in (404, 409, 503):
            raise
        time.sleep(10)
else:
    raise TimeoutError("Expected-version artifact unavailable; no successful readback claim")
if remote.is_dir():
    remote = remote / "probe.zip.bin"
assert remote.resolve() != archive.resolve()
assert hashlib.sha256(remote.read_bytes()).hexdigest() == expected
with zipfile.ZipFile(remote) as stream:
    assert stream.testzip() is None and stream.namelist() == ["probe.json"]
if capabilities["read_kernel_status"] == "COMPLETE":
    request = ApiSaveKernelRequest()
    request.slug = "hungluongduc/hung-api-dispatch-probe-20261027-v1"
    request.new_title = "hung-api-dispatch-probe-20261027-v1"
    request.text = "import json; print(json.dumps({'scope':'CPU dispatch capability probe only','model_training':False}))"
    request.language = "python"
    request.kernel_type = "script"
    request.is_private = True
    request.enable_gpu = False
    request.enable_internet = False
    request.session_timeout_seconds = 300
    with build_kaggle_client() as client:
        try:
            saved = client.kernels.kernels_api_client.save_kernel(request)
            assert not saved.error, saved.error
            capabilities["cpu_child_dispatch"] = {
                "url": saved.url,
                "version_number": saved.version_number,
            }
        except HTTPError as error:
            if error.response.status_code != 403:
                raise
            capabilities["cpu_child_dispatch"] = (
                "FORBIDDEN; no retry or alternate dispatch endpoint"
            )
result = {
    "storage_readback_passed": True,
    "handle": handle,
    "version": 1,
    "privacy": "Official kagglehub create request is_private=True; independent UI confirmation required",
    "sha256": expected,
    "bytes": archive.stat().st_size,
    "crc": "passed",
    "upload_seconds": upload_seconds,
    "http_readback_seconds": time.perf_counter() - readback_started,
    "quota": quota,
    "capabilities": capabilities,
    "kagglehub_version": kagglehub.__version__,
    "authentication": "Existing built-in Kaggle notebook authentication; no credentials printed, copied or created",
    "scope": "Storage and optional CPU-only dispatch capability; not scientific or peer acceptance",
}
with (base / "result.json").open("x") as stream:
    json.dump(result, stream, indent=2)
print(json.dumps(result), flush=True)